In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/multiome/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))

tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']

In [ ]:
gene_ontology_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/degs/fail1/gene_ontology.csv'
go_meta = pd.read_csv(gene_ontology_path, sep='\t', index_col=0)
# start_go = go_meta[go_meta['cluster'] == 'Start'].iloc[:10,:]
# fail1_go = go_meta[go_meta['cluster'] == 'Fail1'].iloc[:10,:]
# go_meta = pd.concat([start_go, fail1_go])
go_meta.shape

In [ ]:
go_meta.head()

In [ ]:
go_dict = dict(zip(go_meta['GO'], go_meta['name']))
# go_dict

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/intersect_atac.h5ad')

In [ ]:
adata

In [ ]:
clusters_path = f'{projdir}/csv/clusters/rna_leiden_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['clusters_v7']
adata.obs['coarse'] = clusters.reindex(adata.obs.index)['coarse_v7']

In [ ]:
adata

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
go_terms = go_meta['GO'].unique()
for tmp_go in go_terms:
    tmp_genes = go_meta[go_meta['GO'] == tmp_go]['genes']
    final_genes = [x.split(',') for x in tmp_genes][0]
    sc.tl.score_genes(adata, final_genes, score_name=tmp_go)

In [ ]:
adata.obs.head()

In [ ]:
coarse_summary = pd.DataFrame(index=tmp_order)

for tmp_go in go_terms:
    go_vals = []
    for clust in tmp_order:
        tmp_meta = adata[adata.obs['coarse'] == clust].obs
        go_vals.append(tmp_meta[tmp_go].mean())
    coarse_summary[tmp_go] = go_vals

In [ ]:
coarse_summary

In [ ]:
coarse_summary.columns

In [ ]:
whitelist = ['Start', 'Delayed', 'Fail1']
mask = [x in whitelist for x in coarse_summary.index]
tmp_coarse = coarse_summary[mask]
tmp_coarse.columns = [go_dict.get(x) for x in tmp_coarse.columns]
tmp_coarse.shape

In [ ]:
tmp_coarse

In [ ]:
tmp_coarse.columns

In [ ]:
fig, ax = plt.subplots(1, figsize=(20,4))
ax = sns.heatmap(tmp_coarse, cmap='coolwarm', vmin=-0.5, vmax=1.5)
ax.grid(False)

In [ ]:
# tmp_clusters = sorted(adata.obs['cluster'].unique())
# tmp_clusters.shape
# tmp_clusters

In [ ]:
tmp_clusters = ['Start_1', 'Start_2',
               'Sendai_1', 'Sendai_2',
               'Delayed_1', 'Delayed_2', 'Delayed_3',
              'Fail1_1', 'Fail1_2', 'Fail1_3', 'Fail1_4',
              'Inter1_1', 'Inter1_2', 'Inter1_3', 'Inter1_4','Inter1_5',
              'Inter2_1', 'Inter2_2', 'Inter2_3', 'Inter2_4',
              'Fail2_2', 'Fail2_3', 'Fail2_4', 'Fail2_5',
                'IPS_1', 'IPS_2', 'IPS_3']

In [ ]:
clusters_summary = pd.DataFrame(index=tmp_clusters)

for tmp_go in go_terms:
    go_vals = []
    for clust in tmp_clusters:
        tmp_meta = adata[adata.obs['cluster'] == clust].obs
        go_vals.append(tmp_meta[tmp_go].mean())
    clusters_summary[tmp_go] = go_vals

In [ ]:
whitelist = ['Start', 'Delayed', 'Fail1']
mask = [x.split('_')[0] in whitelist for x in clusters_summary.index]
sub_clusters = clusters_summary[mask]
sub_clusters.columns = [go_dict.get(x) for x in sub_clusters.columns]
sub_clusters.shape

In [ ]:
fig, ax = plt.subplots(1, figsize=(20,4))
ax = sns.heatmap(sub_clusters, cmap='coolwarm', vmin=-0.5, vmax=1.5)
ax.grid(False)

In [ ]:
tmp_dict = {'biological process involved in interspecies interaction between organisms',
 'interspecies interaction biological process'}

In [ ]:
cluster_out = sub_clusters.T
cluster_out['GO'] = cluster_out.index
cluster_out['GO'] = cluster_out['GO'].replace(tmp_dict)
cluster_out.head()

In [ ]:
outdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/final_figures/csv/figure3'
cluster_out.to_csv(f'{outdir}/cluster_gene_ontology.csv', sep='\t')

In [ ]:
coarse_out = tmp_coarse.T
coarse_out['GO'] = coarse_out.index
coarse_out['GO'] = coarse_out['GO'].replace(tmp_dict)
coarse_out.head()

In [ ]:
outdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/final_figures/csv/figure3'
coarse_out.to_csv(f'{outdir}/coarse_gene_ontology.csv', sep='\t')

In [ ]:
!date